In [1]:
import pandas as pd
import numpy as np
import os
import lightgbm as lgb

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import joblib

In [2]:
folder_path = r"C:\Users\mojes\Desktop\cicids"

files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

df_list = []

for file in files:
    print("Loading:", file)
    temp_df = pd.read_csv(os.path.join(folder_path, file))
    df_list.append(temp_df)

cicids_df = pd.concat(df_list, ignore_index=True)

print("Final Shape:", cicids_df.shape)

Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Wednesday-workingHours.pcap_ISCX.csv
Final Shape: (2830743, 79)


In [3]:
cicids_df.columns = cicids_df.columns.str.strip()

In [4]:
import numpy as np

cicids_df.replace([np.inf, -np.inf], np.nan, inplace=True)
cicids_df = cicids_df.fillna(0)

In [5]:
y = (cicids_df['Label'] != 'BENIGN').astype(int)
X = cicids_df.drop(columns=['Label'])

In [6]:
cols_to_remove = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp']

X = X.drop(columns=[col for col in cols_to_remove if col in X.columns])

In [7]:
sample_df = X.copy()
sample_df['label'] = y

sample_df = sample_df.sample(n=300000, random_state=42)  # 3 lakh rows

y = sample_df['label']
X = sample_df.drop(columns=['label'])

In [8]:
from sklearn.preprocessing import LabelEncoder

for col in X.select_dtypes(include=['object', 'string']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
import lightgbm as lgb

cicids_model = lgb.LGBMClassifier(
    n_estimators=150,
    learning_rate=0.05,
    random_state=42
)

cicids_model.fit(X_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 47276, number of negative: 192724
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.127852 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14594
[LightGBM] [Info] Number of data points in the train set: 240000, number of used features: 70
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.196983 -> initscore=-1.405256
[LightGBM] [Info] Start training from score -1.405256


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,150
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [11]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = cicids_model.predict(X_test)

print("CICIDS Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

CICIDS Accuracy: 0.9990333333333333
Confusion Matrix:
 [[48151    30]
 [   28 11791]]


In [12]:
cm = confusion_matrix(y_test, y_pred)

TP = cm[1,1]
TN = cm[0,0]
FP = cm[0,1]
FN = cm[1,0]

print("Detection Rate:", TP/(TP+FN))
print("FPR:", FP/(FP+TN))

Detection Rate: 0.9976309332430832
FPR: 0.0006226520827712169


In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

def print_binary_metrics(y_true, y_pred):
    
    # Basic metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    TN = cm[0,0]
    FP = cm[0,1]
    FN = cm[1,0]
    TP = cm[1,1]
    
    # Additional metrics
    FPR = FP / (FP + TN)
    FNR = FN / (FN + TP)
    Detection_Rate = TP / (TP + FN)
    
    # Print in your format
    print("\n===== BINARY MODEL PERFORMANCE =====")
    print(f"Accuracy: {acc:.6f}")
    print(f"Precision: {prec:.6f}")
    print(f"Recall: {rec:.6f}")
    print(f"F1 Score: {f1:.6f}")
    
    print("\n===== CONFUSION MATRIX VALUES =====")
    print(f"True Positives (TP): {TP}")
    print(f"True Negatives (TN): {TN}")
    print(f"False Positives (FP): {FP}")
    print(f"False Negatives (FN): {FN}")
    
    print("\n===== ADDITIONAL METRICS =====")
    print(f"False Positive Rate (FPR): {FPR:.6f}")
    print(f"False Negative Rate (FNR): {FNR:.6f}")
    print(f"Detection Rate: {Detection_Rate:.6f}")
    
    print("\n===== CLASSIFICATION REPORT =====")
    print(classification_report(y_true, y_pred, target_names=["Normal", "Attack"]))

In [14]:
print_binary_metrics(y_test, y_pred)


===== BINARY MODEL PERFORMANCE =====
Accuracy: 0.999033
Precision: 0.997462
Recall: 0.997631
F1 Score: 0.997547

===== CONFUSION MATRIX VALUES =====
True Positives (TP): 11791
True Negatives (TN): 48151
False Positives (FP): 30
False Negatives (FN): 28

===== ADDITIONAL METRICS =====
False Positive Rate (FPR): 0.000623
False Negative Rate (FNR): 0.002369
Detection Rate: 0.997631

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     48181
      Attack       1.00      1.00      1.00     11819

    accuracy                           1.00     60000
   macro avg       1.00      1.00      1.00     60000
weighted avg       1.00      1.00      1.00     60000



In [15]:
y_multi = cicids_df['Label']

In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_multi = le.fit_transform(y_multi.astype(str))

In [17]:
sample_df = cicids_df.sample(n=300000, random_state=42)

In [18]:
# Binary (optional)
y_binary = (sample_df['Label'] != 'BENIGN').astype(int)

# Multiclass (correct)
y_multi = sample_df['Label']

# Features
X = sample_df.drop(columns=['Label'])

In [19]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_multi = le.fit_transform(y_multi.astype(str))

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_multi, y_test_multi = train_test_split(
    X,
    y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_multi
)

In [21]:
import lightgbm as lgb

cicids_multi_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(set(y_train_multi)),
    n_estimators=150,
    learning_rate=0.05,
    random_state=42
)

cicids_multi_model.fit(X_train, y_train_multi)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.161694 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14584
[LightGBM] [Info] Number of data points in the train set: 240000, number of used features: 70
[LightGBM] [Info] Start training from score -0.219380
[LightGBM] [Info] Start training from score -7.288528
[LightGBM] [Info] Start training from score -3.092427
[LightGBM] [Info] Start training from score -5.624509
[LightGBM] [Info] Start training from score -2.505926
[LightGBM] [Info] Start training from score -6.272502
[LightGBM] [Info] Start training from score -6.181818
[LightGBM] [Info] Start training from score -5.837314
[LightGBM] [Info] Start training from score -11.695247
[LightGBM] [Info] Start training from score -11.289782
[LightGBM] [Info] Start training from score -2.880619
[LightGBM] [Info] Start training 

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,150
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [52]:
y_pred_multi = cicids_multi_model.predict(X_test)

In [51]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import numpy as np

def print_multiclass_metrics(y_true, y_pred):

    # ===== BASIC METRICS =====
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # ===== CONFUSION MATRIX =====
    cm = confusion_matrix(y_true, y_pred)

    # ===== ADDITIONAL METRICS =====
    # True Positives, False Positives, etc. (per class)
    TP = np.diag(cm)
    FP = np.sum(cm, axis=0) - TP
    FN = np.sum(cm, axis=1) - TP
    TN = np.sum(cm) - (FP + FN + TP)

    # Macro averages
    FPR = np.mean(FP / (FP + TN + 1e-10))
    FNR = np.mean(FN / (FN + TP + 1e-10))
    Detection_Rate = np.mean(TP / (TP + FN + 1e-10))

    # ===== PRINT OUTPUT =====
    print("\n===== MULTICLASS MODEL PERFORMANCE =====")
    print(f"Accuracy: {acc:.6f}")
    print(f"Precision (Weighted): {prec:.6f}")
    print(f"Recall (Weighted): {rec:.6f}")
    print(f"F1 Score (Weighted): {f1:.6f}")

    print("\n===== CONFUSION MATRIX =====")
    print(cm)

    print("\n===== ADDITIONAL METRICS =====")
    print(f"False Positive Rate (Macro): {FPR:.6f}")
    print(f"False Negative Rate (Macro): {FNR:.6f}")
    print(f"Detection Rate (Macro Recall): {Detection_Rate:.6f}")

    print("\n===== CLASSIFICATION REPORT =====")
    print(classification_report(y_true, y_pred, zero_division=0))

In [53]:
print_multiclass_metrics(y_test_multi, y_pred_multi)


===== MULTICLASS MODEL PERFORMANCE =====
Accuracy: 0.700050
Precision (Weighted): 0.749653
Recall (Weighted): 0.700050
F1 Score (Weighted): 0.714771

===== CONFUSION MATRIX =====
[[38512     8     9   947  7308   371    15   158    33    52    98     0
      0    44   626]
 [   23     0     0     0     7     0     0     0     0     0    11     0
      0     0     0]
 [  527     0  1182     0  1014     0     0     0     0     0     0     0
      0     0     0]
 [  217     0     0     0     0     0     0     0     0     0     0     0
      0     0     0]
 [ 2598     0     0     0  2298     0     0     0     0     0     0     0
      0     0     0]
 [   99     0     0     0    14     0     0     0     0     0     0     0
      0     0     0]
 [   80     0     0     0    44     0     0     0     0     0     0     0
      0     0     0]
 [  165     0     0     0     0     1     0     9     0     0     0     0
      0     0     0]
 [    0     0     0     0     0     0     0     0     0     

In [28]:
from sklearn.model_selection import RandomizedSearchCV

In [30]:
import lightgbm as lgb

base_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(set(y_train_multi)),
    class_weight='balanced',
    random_state=42
)

In [31]:
base_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(set(y_train_multi)),
    class_weight='balanced',
    random_state=42
)

In [32]:
param_dist = {
    'num_leaves': [31, 50, 70],
    'learning_rate': [0.01, 0.05],
    'n_estimators': [200, 300],
    'max_depth': [-1, 10, 20],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

In [33]:
from sklearn.model_selection import RandomizedSearchCV

In [34]:
search = RandomizedSearchCV(
    base_model,
    param_distributions=param_dist,
    n_iter=10,
    cv=2,
    scoring='f1_weighted',
    verbose=1,
    n_jobs=-1
)

search.fit(X_train, y_train_multi)

print("Best Params:", search.best_params_)

Fitting 2 folds for each of 10 candidates, totalling 20 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.162976 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14584
[LightGBM] [Info] Number of data points in the train set: 240000, number of used features: 70
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start trai

In [36]:
cicids_multi_tuned = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(set(y_train_multi)),
    class_weight='balanced',
    subsample=0.8,
    num_leaves=50,
    n_estimators=200,
    max_depth=10,
    learning_rate=0.05,
    colsample_bytree=0.8,
    random_state=42
)

cicids_multi_tuned.fit(X_train, y_train_multi)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.166928 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14584
[LightGBM] [Info] Number of data points in the train set: 240000, number of used features: 70
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training from score -2.708050
[LightGBM] [Info] Start training fr

,boosting_type,'gbdt'
,num_leaves,50
,max_depth,10
,learning_rate,0.05
,n_estimators,200
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [37]:
y_pred_multi = cicids_multi_tuned.predict(X_test)

print("Tuned Accuracy:", accuracy_score(y_test_multi, y_pred_multi))
print(classification_report(y_test_multi, y_pred_multi))

Tuned Accuracy: 0.9989333333333333
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     48181
           1       0.83      0.93      0.87        41
           2       1.00      1.00      1.00      2723
           3       1.00      1.00      1.00       217
           4       1.00      1.00      1.00      4896
           5       0.99      0.99      0.99       113
           6       0.98      1.00      0.99       124
           7       1.00      1.00      1.00       175
           8       0.00      0.00      0.00         0
           9       1.00      1.00      1.00         1
          10       1.00      1.00      1.00      3366
          11       1.00      1.00      1.00       121
          12       0.76      0.83      0.79        30
          14       0.44      0.33      0.38        12

    accuracy                           1.00     60000
   macro avg       0.86      0.86      0.86     60000
weighted avg       1.00      1.00      1.00  

C:\Users\mojes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\mojes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\mojes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [38]:
print(len(X_train), len(X_test))
print(len(set(y_test_multi)))

240000 60000
13


In [39]:
import joblib

joblib.dump(cicids_multi_tuned, "cicids_multiclass_model.pkl")

['cicids_multiclass_model.pkl']

In [40]:
joblib.dump(X.columns.tolist(), "cicids_features.pkl")

['cicids_features.pkl']

In [41]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

metrics = {
    "accuracy": accuracy_score(y_test_multi, y_pred_multi),
    "precision_weighted": precision_score(y_test_multi, y_pred_multi, average='weighted'),
    "recall_weighted": recall_score(y_test_multi, y_pred_multi, average='weighted'),
    "f1_weighted": f1_score(y_test_multi, y_pred_multi, average='weighted')
}

joblib.dump(metrics, "cicids_metrics.pkl")

C:\Users\mojes\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


['cicids_metrics.pkl']

In [42]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

def print_multiclass_metrics(y_true, y_pred, label_encoder=None):
    
    # Basic metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    print("\n===== MULTICLASS MODEL PERFORMANCE =====")
    print(f"Accuracy: {acc:.6f}")
    print(f"Precision (Weighted): {prec:.6f}")
    print(f"Recall (Weighted): {rec:.6f}")
    print(f"F1 Score (Weighted): {f1:.6f}")
    
    print("\n===== CONFUSION MATRIX =====")
    print(cm)
    
    print("\n===== CLASSIFICATION REPORT =====")
    
    if label_encoder:
        print(classification_report(
            y_true, y_pred,
            target_names=label_encoder.classes_,
            zero_division=0
        ))
    else:
        print(classification_report(y_true, y_pred, zero_division=0))

In [45]:
y_pred_multi = cicids_multi_tuned.predict(X_test)

In [44]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

def print_multiclass_metrics(y_true, y_pred, label_encoder=None):
    
    print("\n===== MULTICLASS MODEL PERFORMANCE =====")
    
    # Basic metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    print(f"Accuracy: {acc:.6f}")
    print(f"Precision: {prec:.6f}")
    print(f"Recall: {rec:.6f}")
    print(f"F1 Score: {f1:.6f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    
    print("\n===== CONFUSION MATRIX =====")
    print(cm)
    
    # Classification Report
    print("\n===== CLASSIFICATION REPORT =====")
    
    if label_encoder is not None:
        print(classification_report(
            y_true,
            y_pred,
            target_names=label_encoder.classes_,
            zero_division=0
        ))
    else:
        print(classification_report(y_true, y_pred, zero_division=0))

In [46]:
print_multiclass_metrics(y_test_multi, y_pred_multi, le)


===== MULTICLASS MODEL PERFORMANCE =====
Accuracy: 0.998933
Precision: 0.998951
Recall: 0.998933
F1 Score: 0.998937

===== CONFUSION MATRIX =====
[[48142     8     3     1    10     1     1     0     1     0    14     0
      0     0]
 [    3    38     0     0     0     0     0     0     0     0     0     0
      0     0]
 [    1     0  2721     0     1     0     0     0     0     0     0     0
      0     0]
 [    0     0     0   216     1     0     0     0     0     0     0     0
      0     0]
 [    2     0     0     0  4894     0     0     0     0     0     0     0
      0     0]
 [    0     0     0     0     0   112     1     0     0     0     0     0
      0     0]
 [    0     0     0     0     0     0   124     0     0     0     0     0
      0     0]
 [    0     0     0     0     0     0     0   175     0     0     0     0
      0     0]
 [    0     0     0     0     0     0     0     0     0     0     0     0
      0     0]
 [    0     0     0     0     0     0     0     0   

ValueError: Number of classes, 14, does not match size of target_names, 15. Try specifying the labels parameter

In [47]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import numpy as np

def print_multiclass_metrics(y_true, y_pred, label_encoder=None):
    
    print("\n===== MULTICLASS MODEL PERFORMANCE =====")
    
    # Basic metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    print(f"Accuracy: {acc:.6f}")
    print(f"Precision: {prec:.6f}")
    print(f"Recall: {rec:.6f}")
    print(f"F1 Score: {f1:.6f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    
    print("\n===== CONFUSION MATRIX =====")
    print(cm)
    
    # Fix for missing classes
    print("\n===== CLASSIFICATION REPORT =====")
    
    labels = np.unique(y_true)  # only present classes
    
    if label_encoder is not None:
        target_names = label_encoder.classes_[labels]
        
        print(classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=target_names,
            zero_division=0
        ))
    else:
        print(classification_report(
            y_true,
            y_pred,
            labels=labels,
            zero_division=0
        ))

In [48]:
print_multiclass_metrics(y_test_multi, y_pred_multi, le)


===== MULTICLASS MODEL PERFORMANCE =====
Accuracy: 0.998933
Precision: 0.998951
Recall: 0.998933
F1 Score: 0.998937

===== CONFUSION MATRIX =====
[[48142     8     3     1    10     1     1     0     1     0    14     0
      0     0]
 [    3    38     0     0     0     0     0     0     0     0     0     0
      0     0]
 [    1     0  2721     0     1     0     0     0     0     0     0     0
      0     0]
 [    0     0     0   216     1     0     0     0     0     0     0     0
      0     0]
 [    2     0     0     0  4894     0     0     0     0     0     0     0
      0     0]
 [    0     0     0     0     0   112     1     0     0     0     0     0
      0     0]
 [    0     0     0     0     0     0   124     0     0     0     0     0
      0     0]
 [    0     0     0     0     0     0     0   175     0     0     0     0
      0     0]
 [    0     0     0     0     0     0     0     0     0     0     0     0
      0     0]
 [    0     0     0     0     0     0     0     0   

In [49]:
def print_per_class_details(y_true, y_pred, label_encoder):
    cm = confusion_matrix(y_true, y_pred)
    
    print("\n===== PER CLASS DETAILS =====")
    
    for i in range(len(cm)):
        TP = cm[i, i]
        FN = sum(cm[i, :]) - TP
        FP = sum(cm[:, i]) - TP
        TN = cm.sum() - (TP + FP + FN)
        
        print(f"\nClass: {label_encoder.classes_[i]}")
        print(f"TP: {TP}")
        print(f"TN: {TN}")
        print(f"FP: {FP}")
        print(f"FN: {FN}")

In [50]:
print_per_class_details(y_test_multi, y_pred_multi, le)


===== PER CLASS DETAILS =====

Class: BENIGN
TP: 48142
TN: 11811
FP: 8
FN: 39

Class: Bot
TP: 38
TN: 59951
FP: 8
FN: 3

Class: DDoS
TP: 2721
TN: 57274
FP: 3
FN: 2

Class: DoS GoldenEye
TP: 216
TN: 59782
FP: 1
FN: 1

Class: DoS Hulk
TP: 4894
TN: 55091
FP: 13
FN: 2

Class: DoS Slowhttptest
TP: 112
TN: 59886
FP: 1
FN: 1

Class: DoS slowloris
TP: 124
TN: 59874
FP: 2
FN: 0

Class: FTP-Patator
TP: 175
TN: 59825
FP: 0
FN: 0

Class: Heartbleed
TP: 0
TN: 59999
FP: 1
FN: 0

Class: Infiltration
TP: 1
TN: 59999
FP: 0
FN: 0

Class: PortScan
TP: 3363
TN: 56620
FP: 14
FN: 3

Class: SSH-Patator
TP: 121
TN: 59879
FP: 0
FN: 0

Class: Web Attack � Brute Force
TP: 25
TN: 59962
FP: 8
FN: 5

Class: Web Attack � Sql Injection
TP: 4
TN: 59983
FP: 5
FN: 8
